# Práctica 2 — Regularización y Optimizadores
**Materia:** Redes Neuronales  
**Maestría en Ciencias en Inteligencia Artificial — UAQ**

## Objetivo
Comparar el efecto de L1, L2 y Dropout frente a técnicas sin regularización, y evaluar el rendimiento de SGD, Adam y RMSprop sobre el mismo problema.

In [ ]:
import tensorflow as tf
from tensorflow.keras import layers, models, regularizers
from tensorflow.keras.datasets import fashion_mnist
from tensorflow.keras.utils import to_categorical
import matplotlib.pyplot as plt
import numpy as np
print(f"TF {tf.__version__}")

In [ ]:
(X_train, y_train), (X_test, y_test) = fashion_mnist.load_data()
X_train = X_train.reshape(-1,784).astype('float32')/255.0
X_test  = X_test.reshape(-1,784).astype('float32')/255.0
y_train_cat = to_categorical(y_train, 10)
y_test_cat  = to_categorical(y_test, 10)
NOMBRES = ['Camiseta','Pantalón','Suéter','Vestido','Abrigo',
           'Sandalia','Camisa','Tenis','Bolsa','Bota']
print(f"Train: {X_train.shape} | Test: {X_test.shape}")

## 2. Modelos con distintas regularizaciones

In [ ]:
def build_model(reg=None, dropout=0.0, name='model'):
    inp = tf.keras.Input(shape=(784,))
    x = layers.Dense(256, activation='relu', kernel_regularizer=reg, name='d1')(inp)
    if dropout > 0: x = layers.Dropout(dropout)(x)
    x = layers.Dense(128, activation='relu', kernel_regularizer=reg, name='d2')(x)
    if dropout > 0: x = layers.Dropout(dropout)(x)
    out = layers.Dense(10, activation='softmax')(x)
    m = models.Model(inp, out, name=name)
    m.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])
    return m

configs = {
    'Sin reg.':  build_model(name='sin_reg'),
    'L2 (1e-4)': build_model(reg=regularizers.l2(1e-4), name='l2'),
    'Dropout 0.4': build_model(dropout=0.4, name='dropout'),
}
historias = {}
for nombre, modelo in configs.items():
    print(f"Entrenando: {nombre}")
    h = modelo.fit(X_train, y_train_cat, epochs=15, batch_size=256,
                   validation_split=0.1, verbose=0)
    historias[nombre] = h

In [ ]:
plt.figure(figsize=(14,5))
for i, (nombre, h) in enumerate(historias.items()):
    ax = plt.subplot(1,3,i+1)
    ax.plot(h.history['accuracy'], label='Train')
    ax.plot(h.history['val_accuracy'], label='Val')
    ax.set_title(nombre); ax.legend(); ax.grid(True, alpha=0.3)
plt.suptitle('Comparación de regularizaciones — Fashion MNIST', y=1.02)
plt.tight_layout(); plt.savefig('regularizacion.png', dpi=120); plt.show()

## 3. Comparación de optimizadores

In [ ]:
opt_configs = {'SGD': tf.keras.optimizers.SGD(0.01, momentum=0.9),
               'Adam': tf.keras.optimizers.Adam(1e-3),
               'RMSprop': tf.keras.optimizers.RMSprop(1e-3)}

opt_historias = {}
for nombre, opt in opt_configs.items():
    m = build_model(dropout=0.3, name=nombre)
    m.compile(optimizer=opt, loss='categorical_crossentropy', metrics=['accuracy'])
    h = m.fit(X_train, y_train_cat, epochs=15, batch_size=256, validation_split=0.1, verbose=0)
    opt_historias[nombre] = h
    acc = max(h.history['val_accuracy'])
    print(f"{nombre:10s} → val_acc máx: {acc*100:.2f}%")

## Conclusiones
- L2 reduce el sobreajuste penalizando pesos grandes; Dropout es más robusto en capas densas.
- Adam converge más rápido que SGD pero puede sobreajustarse más en datasets pequeños.
- La combinación Dropout + Adam mostró el mejor balance entre convergencia y generalización.